#                     Zomato_Sales_Customer_Analysis


## 1. Database Setup and loading tables in the DB

In [ ]:
create database zomato_db; 

In [ ]:
use zomato_db;

## 2. Data Cleaning and Handling Null Values

In [ ]:
select
    *
from
    customers
where
    customer_name is null
    or registration_date is null

In [ ]:
select
    *
from
    deliveries
where
    order_id is null
    or delivery_status is null
    or delivery_time_minutes is null
    or rider_id is null

In [ ]:
UPDATE deliveries
SET delivery_time_minutes = 0
WHERE delivery_time_minutes IS NULL AND delivery_status = 'cancelled';

In [ ]:
select
    *
from
    orders
where
    customer_id is null
    or restaurant_id is null
    or order_time is null
    or order_date is null
    or order_item is null
    or order_status is null
    or total_amount is null

In [ ]:
select
    *
from
    Riders
where
    rider_name is null
    or signup_date is null;

In [ ]:
select
    *
from
    restaurants
where
    restaurant_name is null
    or city is null
    or opening_hours is null;

## 3.Feature Engineering

In [ ]:
set sql_safe_updates=0;
alter table orders
add column Time_of_Day varchar(20);

Update orders 
set time_of_day =(
                 case 
                    when order_time between '00:00:00' and '12:00:00' then 'Morning'
					when order_time between '12:01:00' and '16:00:00' then 'Afternoon'
                    when order_time between '16:01:00' and '19:00:00' then 'Evening'
					else 'Night'
				  end );



In [ ]:
select * from orders limit 5;

## 3.Business Problems Solved

## Q1. Top 5 Most Frequently Ordered Dishes
-- Write a query to find the top 5 most frequently ordered dishes by the customer "Isha Sharma" in the last 2 year.

In [ ]:
select
    c.customer_name,
    o.order_item,
    count(*) as total_orders,
    dense_rank() over(
        order by
            count(*)
    ) as rank
from
    customers as c
    inner join orders as o on c.customer_id = o.customer_id
where
    c.customer_name = 'Isha Sharma'
    and year(o.order_date) >= year(current_date()) -2
group by
    c.customer_name,
    o.order_item
order by
    total_orders desc,
    rank desc
limit
    5

## Q2. Popular Time Slots
-- Identify the time slots during which the most orders are placed, based on 2-hour intervals.

In [ ]:
select 
     case when hour(order_time) between 0 and 1 then '00:00:00 AM-02:00:00AM'
		  when hour(order_time) between 2 and 3 then '02:00:00 AM-04:00:00 AM'
          when hour(order_time) between 4 and 5 then '04:00:00 AM-06:00:00 AM'
          when hour(order_time) between 6 and 7 then '06:00:00 AM-08:00:00 AM'
		  when hour(order_time) between 8 and 9 then '08:00:00 AM-10:00:00 AM'
          when hour(order_time) between 10 and 11 then '10:00:00 AM-12:00:00 PM'
          when hour(order_time) between 12 and 13 then '12:00:00 PM-02:00:00 PM'
          when hour(order_time) between 14 and 15 then '02:00:00 PM-04:00:00 PM'
          when hour(order_time) between 16 and 17 then '04:00:00 PM-06:00:00 PM'
          when hour(order_time) between 18 and 19 then '06:00:00 PM-08:00:00 PM'
          when hour(order_time) between 20 and 21 then '08:00:00 PM-10:00:00 PM'
          when hour(order_time) between 22 and 23 then '10:00:00 PM-00:00:00 AM'
          end as Time_slot,
	count(order_id) as Total_order
    from orders
    group by time_slot
    order by total_order desc
    



## Q3. Order Value Analysis
-- Find the average order value (AOV) per customer who has placed more than 10 orders. -- Return: customer_name, aov (average order value)

In [ ]:
--create index idx_customername on customers(customer_name);
select
    c.customer_id,
    c.customer_name,
    round(avg(o.total_amount), 2) as AOV,
    count(order_id) as total_orders
from
    orders as o
    inner join customers as c on o.customer_id = c.customer_id
group by
    c.customer_id,
    c.customer_name
having
    total_orders > 10

# Q4. High-Value Customers
-- List the customers who have spent more than 14K in total on food orders. -- Return: customer_name, customer_id.

In [ ]:
select
    c.customer_id,
    c.customer_name,
    sum(total_amount) as total_order_amount
from
    customers as c
    inner join orders as o on c.customer_id = o.customer_id
group by
    c.customer_id,
    c.customer_name
having
    total_order_amount >= 14000
order by
    total_order_amount desc

## Q5. Orders Without Delivery
-- Write a query to find orders that were placed but not delivered. -- Return: restaurant_name, city, and the number of not delivered orders.

In [ ]:
select
    r.restaurant_id,
    r.restaurant_name,
    r.city,
    sum(
        case
            when delivery_status in ('returned', 'cancelled') then 1
            else 0
        end
    ) as Not_delivered_orders
from
    orders as o
    inner join restaurants as r on o.restaurant_id = r.restaurant_id
    inner join deliveries as d on o.order_id = d.order_id
where
    o.order_status = 'out for delivery'
group by
    r.restaurant_id,
    r.restaurant_name,
    r.city
order by
    not_delivered_orders desc

-- create index idx_orderstatus on orders(order_status);
-- create index idx_deliverystatus on deliveries(delivery_status);

## Q6. Restaurant Revenue Ranking
-- Rank restaurants by their total revenue from the last 2 year. -- Return: restaurant_name, total_revenue, and their rank within their city.

In [ ]:
select
    r.restaurant_id,
    r.restaurant_name,
    sum(o.total_amount) as total_revenue,
    r.city,
    dense_rank() over(
        partition by r.city
        order by
            sum(o.total_amount) desc
    ) as city_rank
from
    orders as o
    inner join restaurants as r on o.restaurant_id = r.restaurant_id
where
    year(o.order_date) >= year(current_date()) -2
group by
    r.restaurant_id,
    r.restaurant_name,
    r.city

--create index idx_orderdate on orders(order_date);

## Q7. Most Popular Dish by City
-- Identify the most popular dish in each city based on the number of orders.


In [ ]:
with cte as

(select 
        r.city,
        o.order_item as dish,
        count(*) as no_of_orders,
        dense_rank() over(partition by city order by count(o.order_id) desc) as rnk

from orders as o
inner join restaurants as r
on o.restaurant_id =r.restaurant_id
group by o.order_item, r.city
)

select * from cte
where rnk=1
order by no_of_orders desc

## Q8. Customer Churn
-- Find customers who haven’t placed an order in 2025 but did in 2024.

In [ ]:
select c.customer_id,c.customer_name 
from customers as c 
left join orders as o
on c.customer_id=o.customer_id
where  year(o.order_date)='2024' 
and c.customer_id not in 
          ( select c.customer_id from customers as c
          left join orders as o
            on c.customer_id=o.customer_id 
            where year(o.order_date)='2025')
group by c.customer_id,c.customer_name;



## Q9. Cancellation Rate Comparison
-- Calculate the cancellation rate for each restaurant between the 2024 and 2025.

You are calculating cancellation rate per restaurant:

(Not Fulfilled orders in 2024–2025) ÷ (Total orders) × 100

In [ ]:
WITH cte AS (
    SELECT r.Restaurant_id,
           r.restaurant_name,
           COUNT(o.order_id) AS cancellation_count
    FROM orders AS o
    INNER JOIN restaurants AS r
        ON r.restaurant_id = o.restaurant_id
    WHERE o.order_status = 'Not Fulfilled'
      AND YEAR(order_date) BETWEEN 2024 AND 2025
    GROUP BY r.Restaurant_id, r.restaurant_name
),
total_count AS (
    SELECT restaurants.Restaurant_id,
           restaurants.restaurant_name,
           COUNT(order_id) AS total_counts
    FROM orders
    INNER JOIN restaurants
        ON restaurants.restaurant_id = orders.restaurant_id
    GROUP BY restaurants.Restaurant_id, restaurants.restaurant_name
)
SELECT cte.Restaurant_id,
       cte.restaurant_name,
       CONCAT(ROUND(cte.cancellation_count * 100.0 / total_count.total_counts, 2), ' %') AS cancellation_rate
FROM cte
INNER JOIN total_count
    ON cte.Restaurant_id = total_count.Restaurant_id
ORDER BY cancellation_rate DESC;

## Q10. Rider Average Delivery Time
-- Determine each rider's average delivery time.

In [ ]:
WITH cte AS (
    SELECT 
        r.rider_id,
        r.rider_name,
        d.delivery_time_minutes
    FROM riders r
    JOIN deliveries d
      ON r.rider_id = d.rider_id
    JOIN orders o
      ON o.order_id = d.order_id
    WHERE d.delivery_time_minutes IS NOT NULL
)
SELECT 
    rider_id,
    rider_name,
    ROUND(AVG(delivery_time_minutes), 2) AS avg_delivery_time_mins
FROM cte
GROUP BY rider_id, rider_name
ORDER BY avg_delivery_time_mins ASC;   

## Q11. Monthly Restaurant Growth Ratio
-- Calculate each restaurant's growth ratio for monthly, based on the total number of delivered orders since its joining.



In [ ]:
WITH monthly_orders AS (
    SELECT r.restaurant_id,
           r.restaurant_name,
           TO_CHAR(order_date, 'MM/YYYY') AS month_no,
           COUNT(o.order_id) AS total_orders
    FROM restaurants r
    LEFT JOIN orders o
        ON r.restaurant_id = o.restaurant_id
    LEFT JOIN deliveries d
        ON d.order_id = o.order_id
    WHERE d.delivery_status = 'delivered'
    GROUP BY r.restaurant_id, r.restaurant_name, TO_CHAR(order_date, 'MM/YYYY')
),
cte AS (
    SELECT restaurant_id,
           restaurant_name,
           month_no,
           total_orders,
           LAG(total_orders, 1) OVER (PARTITION BY restaurant_id ORDER BY TO_DATE(month_no, 'MM/YYYY')) AS previous_month_order
    FROM monthly_orders
)
SELECT restaurant_id,
       restaurant_name,
       month_no,
       total_orders,
       previous_month_order,
       ROUND(
           CASE 
               WHEN previous_month_order = 0 OR previous_month_order IS NULL THEN NULL
               ELSE ( (total_orders - previous_month_order) / previous_month_order ) * 100
           END
       , 2) AS monthly_growth_ratio
FROM cte
ORDER BY restaurant_id, month_no;

## Q12. Customer Segmentation
-- Segment customers into 'Gold' or 'Silver' groups based on their total spending compared to the average order value (AOV). If a customer's total spending exceeds the AOV, -- label them as 'Gold'; otherwise, label them as 'Silver'. -- Return: The total number of orders and total revenue for each segment.

In [ ]:
WITH overall_aov AS (
    SELECT AVG(total_amount) AS overall_aov
    FROM orders
),
cust_metrics AS (
    SELECT
        customer_id,
        SUM(total_amount) AS total_spending,
        COUNT(order_id) AS total_orders,
        AVG(total_amount) AS customer_aov
    FROM orders
    GROUP BY customer_id
),
cust_seg AS (
    SELECT
        cm.customer_id,
        CASE
            WHEN cm.customer_aov >= oa.overall_aov THEN 'Gold'
            ELSE 'Silver'
        END AS customer_segment,
        cm.total_spending,
        cm.total_orders
    FROM cust_metrics cm
    CROSS JOIN overall_aov oa
)
SELECT
    customer_segment,
    SUM(total_spending) AS total_revenue,
    SUM(total_orders) AS total_orders
FROM cust_seg
GROUP BY customer_segment;


## Q13. Rider Monthly Earnings
-- Calculate each rider's total monthly earnings, assuming they earn 8% of the order amount.

In [ ]:
SELECT 
  r.rider_id,
  r.rider_name,
  TO_VARCHAR(DATE_TRUNC('month', o.order_date), 'MM-YYYY') AS month_no,
  ROUND(SUM(o.total_amount * 0.08), 2) AS Monthly_Earnings
FROM riders r
LEFT JOIN deliveries d 
  ON r.rider_id = d.rider_id AND d.delivery_status = 'delivered'
LEFT JOIN orders o 
  ON o.order_id = d.order_id AND o.order_status = 'completed'
GROUP BY r.rider_id, r.rider_name, TO_VARCHAR(DATE_TRUNC('month', o.order_date), 'MM-YYYY')
ORDER BY r.rider_id, month_no;

## Q14. Rider Ratings Analysis
-- Find the number of 5-star, 4-star, and 3-star ratings each rider has. Riders receive -- ratings based on delivery time: -- ● 5-star: Delivered in less than 30 minutes -- ● 4-star: Delivered between 30 and 45 minutes -- ● 3-star: Delivered after 45 minutes

In [ ]:
WITH delivery_ratings AS (
    SELECT 
        r.rider_id,
        r.rider_name,
        CASE
            WHEN d.delivery_time_minutes < 30 THEN '5-star'
            WHEN d.delivery_time_minutes BETWEEN 30 AND 45 THEN '4-star'
            ELSE '3-star'
        END AS rating
    FROM riders r
    LEFT JOIN deliveries d
        ON r.rider_id = d.rider_id
    WHERE d.delivery_time_minutes IS NOT NULL
)
SELECT 
    rider_id,
    rider_name,
    COUNT(CASE WHEN rating = '5-star' THEN 1 END) AS five_star_rating,
    COUNT(CASE WHEN rating = '4-star' THEN 1 END) AS four_star_rating,
    COUNT(CASE WHEN rating = '3-star' THEN 1 END) AS three_star_rating
FROM delivery_ratings
GROUP BY rider_id, rider_name
ORDER BY rider_id;

## Q15. Order Frequency by Day
-- Analyze order frequency per day of the week and identify the peak day for each restaurant.

In [ ]:
with cte as 
(
select r.restaurant_id, r.restaurant_name,r.city,
       count(o.order_id) as total_orders,
       dayname(o.order_date) as day_of_week
from orders as o
inner join restaurants as r
on o.restaurant_id=r.restaurant_id
group by r.restaurant_id, r.restaurant_name,r.city, day_of_week
order by r.restaurant_id, r.restaurant_name,r.city
),
highest_order_day as
(
select restaurant_id, restaurant_name,city, total_orders,day_of_week,
dense_rank() over(partition by restaurant_id order by total_orders desc) as rnk
from cte
)

select  restaurant_id, restaurant_name,city, total_orders,day_of_week
from highest_order_day
where rnk=1

In [ ]:
WITH cte AS (
  SELECT 
    r.restaurant_id, 
    r.restaurant_name,
    r.city,
    COUNT(o.order_id) AS total_orders,
    DAYNAME(o.order_date) AS day_of_week
  FROM orders o
  INNER JOIN restaurants r ON o.restaurant_id = r.restaurant_id
  GROUP BY r.restaurant_id, r.restaurant_name, r.city, DAYNAME(o.order_date)
),
highest_order_day AS (
  SELECT 
    restaurant_id, 
    restaurant_name,
    city, 
    total_orders, 
    day_of_week,
    DENSE_RANK() OVER (PARTITION BY restaurant_id ORDER BY total_orders DESC) AS rnk
  FROM cte
)

SELECT 
  restaurant_id, 
  restaurant_name,
  city, 
  total_orders, 
  day_of_week
FROM highest_order_day
WHERE rnk = 1
order by total_orders desc;

## Q16. Customer Lifetime Value (CLV)
-- Calculate the total revenue generated by each customer over all their orders.

In [ ]:
select c.customer_id, c.customer_name,
       sum(o.total_amount) as CLV
from customers as c
left join orders as o
on c.customer_id = o.customer_id
group by c.customer_id, c.customer_name

## Q17. Monthly Sales Trends
-- Identify sales trends by comparing each month's total sales to the previous month.



In [ ]:
with cte as (
select 
        TO_VARCHAR(DATE_TRUNC('year', order_date), 'YYYY') AS year_no,
        TO_VARCHAR(DATE_TRUNC('month', order_date), 'MM') AS month_no,
        sum(total_amount) as total_sales,
        lag(sum(total_amount),1) over(order by TO_VARCHAR(DATE_TRUNC('year', order_date), 'YYYY'), 
        TO_VARCHAR(DATE_TRUNC('month', order_date), 'MM')  ) as prevoius_month_sales
from orders 
group by year_no,month_no
)
select year_no,month_no,total_sales, prevoius_month_sales
from cte 
order by year_no,month_no;


## Q18. Rider Efficiency
-- Evaluate rider efficiency by determining average delivery times and identifying those with the lowest and highest averages.

In [ ]:
WITH cte AS (
    SELECT 
        r.rider_id,
        r.rider_name,
        d.delivery_time_minutes
    FROM riders r
    JOIN deliveries d
        ON r.rider_id = d.rider_id
    JOIN orders o
        ON o.order_id = d.order_id
    WHERE o.order_status = 'completed'
      AND d.delivery_status = 'delivered'
      AND d.delivery_time_minutes IS NOT NULL
)
, avg_deliv_time AS (
    SELECT 
        rider_id,
        rider_name,
        ROUND(AVG(delivery_time_minutes), 2) AS avg_delivery_time_mins
    FROM cte
    GROUP BY rider_id, rider_name
)
SELECT *
FROM avg_deliv_time
ORDER BY avg_delivery_time_mins ASC;

## Q19. Order Item Popularity
-- Track the popularity of specific order items over time and identify seasonal demand spikes.

In [ ]:
 with cte as(
 select order_item,count(order_id) as Total_orders,
    case 
       when month(order_date) between 3 and 5 then 'Spring'
       when month(order_date) between 6 and 8 then 'Summer'
       when month(Order_date) between 9 and 11 then 'Autumn'
       else 'Winter'
       end as Seasons
from orders 
where order_status='completed'
group by order_item,seasons),

Trending_item as (
select  order_item,Total_orders, seasons,dense_rank() over(partition by order_item order by total_orders desc) as rnk
from cte)

select order_item,Total_orders,seasons from trending_item where rnk=1
order by total_orders desc;

## Q20. City Revenue Ranking
-- Rank each city based on the total revenue for the last year (2025).

In [ ]:
SELECT
    r.city,
    COALESCE(SUM(o.total_amount), 0) AS total_revenue,
    DENSE_RANK() OVER (ORDER BY COALESCE(SUM(o.total_amount), 0) DESC) AS rnk
FROM restaurants r
LEFT JOIN orders o
    ON r.restaurant_id = o.restaurant_id
    AND YEAR(o.order_date) = 2025
GROUP BY r.city;

## Stored Procedure

1. create a stored procedure to show the full order history of a customer sorted by most recent -- return customer_name,order_id,restaurant_name, order_date,total amount

In [ ]:
CREATE OR REPLACE PROCEDURE customer_details(p_customer_id STRING)
RETURNS TABLE (
    customer_name STRING,
    order_id STRING,
    restaurant_name STRING,
    order_date DATE,
    total_amount INT
)
LANGUAGE SQL
AS
$$
DECLARE
    res RESULTSET;
BEGIN
    res := (
        SELECT 
            c.customer_name,
            o.order_id,
            r.restaurant_name,
            o.order_date,
            o.total_amount
        FROM customers c
        LEFT JOIN orders o 
            ON c.customer_id = o.customer_id
        LEFT JOIN restaurants r 
            ON r.restaurant_id = o.restaurant_id
        WHERE c.customer_id = :p_customer_id
        ORDER BY o.order_date DESC
    );

    RETURN TABLE(res);
END;
$$;


In [ ]:
SELECT *
FROM TABLE(customer_details('CUST001'));
